In [ ]:
import pandas as pd
from statsmodels.tsa.seasonal import STL
import numpy as np
import matplotlib.pyplot as plt
import random
from typing import List, Tuple
from sklearn.metrics import mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.ar_model import AutoReg

def gerar_cor_aleatoria():
    return (random.random(), random.random(), random.random())


In [ ]:
df = pd.read_csv('AirPassengers.csv', parse_dates=['Month'])
serie = df['#Passengers']

plt.figure(figsize=(12,5))
plt.plot(serie)
plt.title('Série Original')
plt.show()



In [ ]:
# ── ADF + KPSS: Série Original ──────────────────────────────────────────────
def teste_estacionariedade(serie, label='Série'):
    print(f'{'─'*55}')
    print(f'  Testes de Estacionariedade — {label}')
    print(f'{'─'*55}')

    # ADF — H0: raiz unitária (não estacionária)
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(serie, autolag='AIC')
    print(f'\nADF')
    print(f'  Estatística : {adf_stat:.4f}')
    print(f'  p-value     : {adf_p:.4f}')
    for k, v in adf_crit.items():
        print(f'  Valor crítico {k}: {v:.4f}')
    adf_concl = 'Estacionária (rejeita H0)' if adf_p < 0.05 else 'Não estacionária (falha em rejeitar H0)'
    print(f'  → {adf_concl}')

    # KPSS — H0: estacionária
    kpss_stat, kpss_p, _, kpss_crit = kpss(serie, regression='c', nlags='auto')
    print(f'\nKPSS')
    print(f'  Estatística : {kpss_stat:.4f}')
    print(f'  p-value     : {kpss_p:.4f}')
    for k, v in kpss_crit.items():
        print(f'  Valor crítico {k}: {v:.4f}')
    kpss_concl = 'Não estacionária (rejeita H0)' if kpss_p < 0.05 else 'Estacionária (falha em rejeitar H0)'
    print(f'  → {kpss_concl}')
    print()

teste_estacionariedade(serie, 'Série Original')


In [ ]:
# ── ACF + PACF: Série Original ───────────────────────────────────────────────
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(serie, lags=40, ax=ax[0])
ax[0].set_title('ACF — Série Original')
plot_pacf(serie, lags=40, ax=ax[1], method='ywm')
ax[1].set_title('PACF — Série Original')
plt.tight_layout()
plt.show()


In [ ]:
serie_diff = serie.diff().dropna()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(serie_diff)
plt.title('Série Diferenciada')
plt.show()

In [ ]:
# ── ACF + PACF: Série Diferenciada ───────────────────────────────────────────
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(serie_diff, lags=40, ax=ax[0])
ax[0].set_title('ACF — Série Diferenciada')
plot_pacf(serie_diff, lags=40, ax=ax[1], method='ywm')
ax[1].set_title('PACF — Série Diferenciada')
plt.tight_layout()
plt.show()


In [ ]:
# ── ADF + KPSS: Série Diferenciada ───────────────────────────────────────────
teste_estacionariedade(serie_diff, 'Série Diferenciada')


In [ ]:
# ── Base Model 1: Média Histórica (Historical Mean) ──────────────────────────
def gerando_media_historica(serie: pd.Series, test_size: int = 24) -> Tuple[pd.Series, float]:
    """
    Média histórica one-step-ahead.
    Parâmetros
    ----------
    serie     : série completa (pd.Series)
    test_size : número de pontos no conjunto de teste
    Retorna
    -------
    previsao_teste : previsões no período de teste
    mae            : MAE no conjunto de teste
    """
    n = len(serie)
    train = serie.iloc[:n - test_size]
    test  = serie.iloc[n - test_size:]

    # One-step-ahead: a cada passo usa apenas o histórico até aquele ponto
    previsao_full = pd.Series(
        [serie.iloc[:t].mean() for t in range(1, n)],
        index=serie.index[1:],
        dtype=float
    )
    previsao_teste = previsao_full.loc[test.index]

    mae = mean_absolute_error(test, previsao_teste)

    # Plot
    plt.figure(figsize=(12, 5))
    plt.plot(train, label='Treino', color='steelblue')
    plt.plot(test,  label='Teste (real)', color='black')
    plt.plot(previsao_teste, label=f'Média Histórica (MAE: {mae:.2f})',
             color='red', linestyle='--')
    plt.axvline(train.index[-1], color='gray', linestyle=':', label='Início do Teste')
    plt.legend()
    plt.title('Base Model: Média Histórica')
    plt.tight_layout()
    plt.show()

    print(f'MAE Média Histórica (teste): {mae:.4f}')
    return previsao_teste, mae

prev_historica, mae_historica = gerando_media_historica(serie, test_size=24)


In [ ]:
# ── Base Model 2: Média Acumulada (Expanding Mean) ───────────────────────────
def gerando_media_acumulada(serie: pd.Series, test_size: int = 24) -> Tuple[pd.Series, float]:
    """
    Média acumulada (expanding window) one-step-ahead.
    Retorna
    -------
    previsao_teste : previsões no período de teste
    mae            : MAE no conjunto de teste
    """
    n = len(serie)
    train = serie.iloc[:n - test_size]
    test  = serie.iloc[n - test_size:]

    # shift(1) garante que cada previsão usa apenas dados anteriores
    previsao_full  = serie.expanding().mean().shift(1).dropna()
    previsao_teste = previsao_full.loc[test.index]

    mae = mean_absolute_error(test, previsao_teste)

    plt.figure(figsize=(12, 5))
    plt.plot(train, label='Treino', color='steelblue')
    plt.plot(test,  label='Teste (real)', color='black')
    plt.plot(previsao_teste, label=f'Média Acumulada (MAE: {mae:.2f})',
             color='green', linestyle='--')
    plt.axvline(train.index[-1], color='gray', linestyle=':', label='Início do Teste')
    plt.legend()
    plt.title('Base Model: Média Acumulada')
    plt.tight_layout()
    plt.show()

    print(f'MAE Média Acumulada (teste): {mae:.4f}')
    return previsao_teste, mae

prev_acumulada, mae_acumulada = gerando_media_acumulada(serie, test_size=24)


In [ ]:
# ── Base Model 3: Média Móvel Simples (SMA) ──────────────────────────────────
def gerando_media_movel(serie: pd.Series, test_size: int = 24,
                         max_janela: int = 24) -> Tuple[pd.Series, float, int]:
    """
    Média Móvel Simples com busca pela melhor janela no conjunto de treino.
    Parâmetros
    ----------
    serie      : série completa
    test_size  : pontos no conjunto de teste
    max_janela : janela máxima a testar (default: 24)
    Retorna
    -------
    previsao_teste : previsões no período de teste com a melhor janela
    mae            : MAE no conjunto de teste
    best_k         : melhor janela encontrada no treino
    """
    n = len(serie)
    train = serie.iloc[:n - test_size]
    test  = serie.iloc[n - test_size:]

    # ── Seleção de janela no treino ──
    resultados = []
    for janela in range(1, max_janela + 1):
        rolling = train.rolling(window=janela).mean().shift(1)
        validos = rolling.dropna()
        if len(validos) == 0:
            continue
        mae_tr = mean_absolute_error(train[validos.index], validos)
        resultados.append({'Janela': janela, 'MAE_treino': mae_tr})

    df_res = pd.DataFrame(resultados).sort_values('MAE_treino')
    best_k = int(df_res.iloc[0]['Janela'])
    print(f'Melhor janela (selecionada no treino): {best_k}')
    print(df_res.to_string(index=False))

    # ── Previsão no teste com melhor janela ──
    # One-step-ahead no teste: usa janela de 'best_k' pontos anteriores (do histórico completo)
    previsao_full  = serie.rolling(window=best_k).mean().shift(1)
    previsao_teste = previsao_full.loc[test.index]

    mae = mean_absolute_error(test, previsao_teste)

    # ── Plot comparativo de janelas no treino ──
    plt.figure(figsize=(12, 5))
    plt.plot(train, label='Treino', color='steelblue', linewidth=2)
    for row in resultados:
        j = row['Janela']
        rm = train.rolling(window=j).mean().shift(1)
        plt.plot(rm, color=gerar_cor_aleatoria(), alpha=0.4,
                 label=f'SMA {j} (MAE={row["MAE_treino"]:.1f})')
    plt.title('SMA — Comparação de Janelas (Treino)')
    plt.tight_layout()
    plt.show()

    # ── Plot final: melhor janela no teste ──
    plt.figure(figsize=(12, 5))
    plt.plot(train, label='Treino', color='steelblue')
    plt.plot(test,  label='Teste (real)', color='black')
    plt.plot(previsao_teste, label=f'SMA k={best_k} (MAE: {mae:.2f})',
             color='orange', linestyle='--')
    plt.axvline(train.index[-1], color='gray', linestyle=':', label='Início do Teste')
    plt.legend()
    plt.title(f'Base Model: Média Móvel Simples (melhor janela k={best_k})')
    plt.tight_layout()
    plt.show()

    print(f'MAE SMA k={best_k} (teste): {mae:.4f}')
    return previsao_teste, mae, best_k

prev_sma, mae_sma, best_k = gerando_media_movel(serie, test_size=24, max_janela=24)


## Comparação dos Base Models


In [ ]:
# ── Comparação final dos Base Models ─────────────────────────────────────────
resultados_base = pd.DataFrame({
    'Modelo': ['Média Histórica', 'Média Acumulada', f'SMA k={best_k}'],
    'MAE (teste)': [mae_historica, mae_acumulada, mae_sma]
}).sort_values('MAE (teste)')

print(resultados_base.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.barh(resultados_base['Modelo'], resultados_base['MAE (teste)'], color=['#e74c3c','#2ecc71','#f39c12'])
plt.xlabel('MAE (conjunto de teste)')
plt.title('Comparação Base Models')
plt.tight_layout()
plt.show()
